In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  2
me:  2
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  2
me:  2
trying: ['responses_df_2017']
me:  1
trying: ['px']


me:  0
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['warnings']
me:  0
trying: ['responses_df_2019_cell10']
me:  2
trying: ['np']
me:  0
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['percentages_per_country_df']
me:  4
trying: ['file_name_2021']
me:  1
trying: ['orientation_for_chart']
me:  5
trying: ['load_survey_data']
me:  1
trying: ['responses_in_order']
me:  5
trying: ['factor']
me:  1
trying: ['percentages']
me:  5
trying: ['go']
me:  0
trying: ['directory']
me:  1
trying: ['question_of_interest']
me:  5
trying: ['sns']
me:  0
trying: ['file_name_2018']
me:  1
trying: ['country']
me:  5
trying: ['glob']
me:  0
trying: ['base_dir_2022']
me:  1
trying: ['percentages_cell17']
me:  5
trying: ['file_name_2019']
me:  1
trying: ['pd']
me:  0
trying: ['file_name_2022']
me:  1
trying: ['base_dir_2019']
me:  1
trying: ['base_dir_2018']
me:  1
trying: ['base_dir_2021']
me:  1
trying: ['base_dir_2017']
me:  1
trying: ['file_name_2020']
me:  1
trying: ['file_name_2017'

me:  2
trying: ['base_dir_2020']
me:  1
Declaring variable px
Declaring variable warnings
Declaring variable np
Declaring variable go
Declaring variable sns
Declaring variable glob
Declaring variable pd
Declaring variable responses_df_2017
Declaring variable file_name_2021
Declaring variable load_survey_data
Declaring variable factor
Declaring variable directory
Declaring variable file_name_2018
Declaring variable base_dir_2022
Declaring variable file_name_2019
Declaring variable file_name_2022
Declaring variable base_dir_2019
Declaring variable base_dir_2018
Declaring variable base_dir_2021
Declaring variable base_dir_2017
Declaring variable file_name_2020
Declaring variable file_name_2017
Declaring variable responses_df_2019
Declaring variable responses_df_2020
Declaring variable responses_df_2018
Declaring variable directory_cell8
Declaring variable responses_df_2021
Declaring variable base_dir_2020
Declaring variable responses_df_2022_cell10
Declaring variable responses_df_2022
Dec

In [3]:
%%RecordEvent
%%time
### cell 6 ###

def count_then_return_percent_18(df, col):
    # faster: use normalize in value_counts
    return df[col].value_counts(dropna=False, normalize=True).mul(100).round(1)


def combine_subset_of_data_from_multiple_years_18(question_of_interest, x_axis_title, include_2017=False):
    # map each year to its source DataFrame (no in-place modifications)
    df_map = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10,
    }
    if include_2017:
        # rename 2017 columns on-the-fly
        df_map['2017'] = responses_df_2017.rename(columns={
            'Country': question_of_interest,
            'CurrentJobTitleSelect': 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
        })

    rows = []
    for year in sorted(df_map.keys()):
        df_y = df_map[year]
        # 2022: rename UK column and values
        if year == '2022':
            df_y = df_y.rename(columns={
                'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'
            })
            df_y = df_y.assign(**{
                question_of_interest: df_y[question_of_interest].replace({
                    'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'
                })
            })
        # 2017: standardize country values
        if year == '2017':
            df_y = df_y.assign(**{
                question_of_interest: df_y[question_of_interest].replace({
                    'United States': 'United States of America',
                    "People 's Republic of China": 'China',
                    'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
                })
            })
        # collapse to subset_of_countries, label others 'Other'
        df_y = df_y.assign(**{
            question_of_interest: df_y[question_of_interest].where(
                df_y[question_of_interest].isin(subset_of_countries), 'Other'
            )
        })
        # compute percent and build year DataFrame
        pct = count_then_return_percent_18(df_y, question_of_interest).sort_index()
        df_pct = pct.reset_index().rename(columns={'index': question_of_interest, question_of_interest: 'percentage'})
        df_pct['year'] = year
        rows.append(df_pct)

    combined = pd.concat(rows, ignore_index=True)
    # rename the x-axis column if needed
    if x_axis_title:
        combined = combined.rename(columns={question_of_interest: x_axis_title})
    return combined

# prepare parameters
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]
question_of_interest_cell18 = 'In which country do you currently reside?'
# empty string for x-axis title as before
title_for_x_axis = ''

# get combined DataFrame
country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_of_interest_cell18,
    title_for_x_axis,
    include_2017=True
)
# sort and finalize naming
country_df_combined_cell18 = country_df_combined.sort_values(['year', 'percentage'], ascending=True)
# replace long UK name with shorter
country_df_combined_cell18 = country_df_combined_cell18.replace(
    'United Kingdom of Great Britain and Northern Ireland',
    'United Kingdom'
)
country_df_combined_cell18.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  78 non-null     object 
 1   proportion  78 non-null     float64
 2   year        78 non-null     object 
dtypes: float64(1), object(2)
memory usage: 2.0+ KB
CPU times: user 93 ms, sys: 4.15 ms, total: 97.2 ms
Wall time: 96.9 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_6_try_0.pickle

migration speed (bps): 752784969.4823362


---------------------------
variables to migrate:
responses_df_2022_cell10 25594283
percentages 958
question_of_interest 90
create_dataframe_of_counts_16 144
country 174012
percentages_per_country_df 4474
percentages_cell17 958
orientation_for_chart 50
responses_in_order 184
responses_df_2017 15718710
file_name_2021 81
title_for_x_axis 49
load_survey_data 144
factor 24
directory 163
question_of_interest_cell18 90
file_name_2018 76
country_df_combined_cell18 10554
base_dir_2022 155
country_df_combined 10554
file_name_2019 78
file_name_2022 81
base_dir_2019 155
base_dir_2018 155
combine_subset_of_data_from_multiple_years_18 144
base_dir_2021 155
subset_of_countries 168
base_dir_2017 155
file_name_2020 81
file_name_2017 76
responses_df_2019 16501304
responses_df_2020 25593902
responses_df_2018 32983443
directory_cell8 170
replace_hyphen_with_en_dash 144
responses_df_2021 34358547
responses_df_2019_cell10 16059576
base_dir_2020 155
responses_df_2018_cell10 32150664
count_then_return_percen

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2020', 'responses_df_2019', 'responses_df_2018', 'responses_df_2021', 'responses_df_2022']
Intermediate variables ['base_dir_2017', 'file_name_2017', 'file_name_2022', 'file_name_2020', 'base_dir_2020', 'directory_cell8', 'base_dir_2019', 'factor', 'base_dir_2018', 'file_name_2019', 'file_name_2021', 'base_dir_2021', 'responses_df_2017', 'base_dir_2022', 'directory', 'file_name_2018']
Future variables ['responses_df_2022_cell10', 'responses_df_2017']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2019', 'responses_df_2018', 'responses_df_2022']
Active variables ['responses_df_2022', 'responses_df_2019_cell10']
Intermediate variables ['responses_df_2018_cell10', 'responses_df_2022_cell10']
Future variables ['responses_df_2021', 'responses_df_2022_cell10', 'respo

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_6_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[6], f)


In [7]:
opt_output = Out.get(4)

In [8]:
title_for_x_axis_opt = title_for_x_axis
subset_of_countries_opt = subset_of_countries
responses_df_2020_opt = responses_df_2020
responses_df_2021_opt = responses_df_2021
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_6.pickle
assert title_for_x_axis_opt == title_for_x_axis
assert subset_of_countries_opt == subset_of_countries

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['responses_df_2022', 'responses_df_2022_cell10']


me:  10
me:  10
trying: ['responses_df_2022', 'responses_df_2022_cell10']


me:  10
me:  10
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['file_name_2018', 'file_name_2017']
me:  2
me:  2
trying: ['file_name_2018', 'file_name_2017']
me:  2
me:  2
trying: ['percentages_cell17']
me:  10
trying: ['base_dir_2022']
me:  2
trying: ['add_year_column_to_dataframes_18']
me:  12
trying: ['count_then_return_percent_17']
me:  10
trying: ['base_dir_2017']
me:  2
trying: ['file_name_2020']
me:  2
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  12
trying: ['glob']
me:  0
trying: ['go']
me:  0
trying: ['country_df_combined_cell18']
me:  12
trying: ['px']
me:  0
trying: ['sns']
me:  0
trying: ['pd']
me:  0
trying: ['responses_df_2019_cell10']


me:  12
trying: ['np']
me:  0
trying: ['responses_df_2021']


me:  12
trying: ['warnings']
me:  0
trying: ['orig_output']
me:  1
trying: ['responses_df_2018_cell10']


...

me:  12
trying: ['responses_df_2019']


me:  2
trying: ['responses_df_2018']


me:  2
trying: ['subset_of_countries']
me:  12
trying: ['replace_hyphen_with_en_dash']
me:  4
trying: ['create_dataframe_of_counts_16']
me:  8
trying: ['percentages_per_country_df']
me:  8
trying: ['count_then_return_percent_18']
me:  12
trying: ['responses_df_2020']


me:  12
trying: ['base_dir_2018']
me:  2
trying: ['question_name_alternate']
me:  12
trying: ['file_name_2022']
me:  2
trying: ['file_name_2021']
me:  2
trying: ['directory']
me:  2
trying: ['base_dir_2021']
me:  2
trying: ['file_name_2019']
me:  2
trying: ['percentages']
me:  10
trying: ['load_survey_data']
me:  2
trying: ['responses_in_order']
me:  10
trying: ['base_dir_2020']
me:  2
trying: ['country_df_combined']
me:  12
trying: ['orientation_for_chart']
me:  10
trying: ['directory_cell8']
me:  2
trying: ['question_of_interest']
me:  10
trying: ['base_dir_2019']
me:  2
trying: ['factor']
me:  2


Declaring variable glob
Declaring variable go
Declaring variable px
Declaring variable sns
Declaring variable pd
Declaring variable np
Declaring variable warnings
Declaring variable orig_output
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable base_dir_2022
Declaring variable base_dir_2017
Declaring variable file_name_2020
Declaring variable responses_df_2019
Declaring variable responses_df_2018
Declaring variable base_dir_2018
Declaring variable file_name_2022
Declaring variable file_name_2021
Declaring variable directory
Declaring variable base_dir_2021
Declaring variable file_name_2019
Declaring variable load_survey_data
Declaring variable base_dir_2020
Declaring variable directory_cell8
Declaring variable base_dir_2019
Declaring variable factor
Declaring variable replace_hyphen_with_en_dash
Declaring variable create_dataframe_of_counts_16
Declaring variable percentages_per_count